# Notebook 06: Hybrid E-Commerce Recommendation System
### Combining Content-Based Filtering (Notebook 04) + Collaborative Filtering (Notebook 05)

**Scope of this notebook:** combine the already-trained content-based and collaborative models
into a single hybrid recommender, tune the blend weight, evaluate against both individual models
and a popularity baseline, and save the final recommendation engine. **No API or deployment work
happens here** — that's a later notebook.


---
## 1. Load Previous Models

We reuse everything already trained — nothing here retrains TF-IDF or ALS from scratch. This
cell auto-detects paths and includes a lightweight fallback: if the Content-Based artifacts
(`cb_artifacts/`) aren't found (e.g. they were only saved to a local Colab session that has since
ended, rather than Drive), it rebuilds the TF-IDF vectorizer directly from `content_features.csv`
— this is fast (seconds, not minutes) and produces an identical result, since it's the same
deterministic config from Notebook 04. It does **not** retrain ALS, which is the expensive part.


In [1]:
import os

PROCESSED_DATA_DIR = "processed_data"

if not os.path.exists(os.path.join(PROCESSED_DATA_DIR, "train_interactions_encoded.parquet")):
    print("Files not found locally — mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    PROCESSED_DATA_DIR = "/content/drive/MyDrive/hm_recsys/processed_data"

BASE_DIR = os.path.join(PROCESSED_DATA_DIR, "..")
MODELS_DIR = os.path.join(BASE_DIR, "models")
ARTIFACTS_DIR = os.path.join(BASE_DIR, "artifacts")
CB_ARTIFACTS_DIR = "cb_artifacts"  # Notebook 04 default; adjust if you saved it elsewhere

print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)
print("MODELS_DIR:", MODELS_DIR)
print("ARTIFACTS_DIR:", ARTIFACTS_DIR)
print("CB_ARTIFACTS_DIR:", CB_ARTIFACTS_DIR)


Files not found locally — mounting Google Drive...
Mounted at /content/drive
PROCESSED_DATA_DIR: /content/drive/MyDrive/hm_recsys/processed_data
MODELS_DIR: /content/drive/MyDrive/hm_recsys/processed_data/../models
ARTIFACTS_DIR: /content/drive/MyDrive/hm_recsys/processed_data/../artifacts
CB_ARTIFACTS_DIR: cb_artifacts


**1.1 Load Content-Based artifacts (with quick-rebuild fallback)**

In [2]:
import pandas as pd
import pickle
from scipy.sparse import load_npz
from sklearn.feature_extraction.text import TfidfVectorizer

content_features = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, "content_features.csv"))
processed_articles = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, "processed_articles.csv"))

cb_vectorizer_path = os.path.join(CB_ARTIFACTS_DIR, "tfidf_vectorizer.pkl")
cb_matrix_path = os.path.join(CB_ARTIFACTS_DIR, "tfidf_matrix.npz")

if os.path.exists(cb_vectorizer_path) and os.path.exists(cb_matrix_path):
    with open(cb_vectorizer_path, "rb") as f:
        tfidf_vectorizer = pickle.load(f)
    tfidf_matrix = load_npz(cb_matrix_path)
    print("Loaded existing Content-Based artifacts.")
else:
    print("Content-Based artifacts not found locally — rebuilding TF-IDF quickly "
          "(same deterministic config as Notebook 04, seconds not minutes)...")
    FINAL_TFIDF_CONFIG = {"max_features": 10000, "min_df": 2, "max_df": 0.8, "ngram_range": (1, 2)}
    tfidf_vectorizer = TfidfVectorizer(**FINAL_TFIDF_CONFIG, stop_words="english")
    tfidf_matrix = tfidf_vectorizer.fit_transform(content_features["combined_features"].fillna(""))
    print("Rebuilt TF-IDF matrix:", tfidf_matrix.shape)

cb_article_ids = content_features["article_id"].values
cb_id_to_index = {aid: idx for idx, aid in enumerate(cb_article_ids)}

print("Content-Based TF-IDF matrix shape:", tfidf_matrix.shape)


Content-Based artifacts not found locally — rebuilding TF-IDF quickly (same deterministic config as Notebook 04, seconds not minutes)...
Rebuilt TF-IDF matrix: (105542, 10000)
Content-Based TF-IDF matrix shape: (105542, 10000)


**1.2 Load Collaborative Filtering artifacts**

In [3]:
!pip install -q implicit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 52.2 MB/s eta 0:00:00


In [4]:
try:
    import implicit
except ModuleNotFoundError:
    print("Installing implicit library...")
    !pip install -q implicit
    import implicit

with open(os.path.join(MODELS_DIR, "als_model.pkl"), "rb") as f:
    als_bundle = pickle.load(f)
als_model = als_bundle["model"]

In [5]:
with open(os.path.join(MODELS_DIR, "als_model.pkl"), "rb") as f:
    als_bundle = pickle.load(f)
als_model = als_bundle["model"]

with open(os.path.join(PROCESSED_DATA_DIR, "label_encoders.pkl"), "rb") as f:
    encoders = pickle.load(f)
user_encoder = encoders["user_encoder"]
item_encoder = encoders["item_encoder"]

train_matrix = load_npz(os.path.join(ARTIFACTS_DIR, "train_matrix.npz"))

with open(os.path.join(ARTIFACTS_DIR, "popularity_baseline.pkl"), "rb") as f:
    pop_bundle = pickle.load(f)
popularity_ranking = pop_bundle["popularity_ranking"]

print("ALS model config:", als_bundle["config"])
print("Train matrix shape:", train_matrix.shape)
print("Users known to CF:", len(user_encoder.classes_))
print("Items known to CF:", len(item_encoder.classes_))


ALS model config: {'factors': 32, 'regularization': 0.01, 'iterations': 20}
Train matrix shape: (1362281, 104547)
Users known to CF: 1362281
Items known to CF: 104547


**1.3 Verify ID consistency between the two models**

The Content-Based index space (`cb_id_to_index`, built from `content_features.csv`) and the
Collaborative Filtering index space (`item_encoder`, built from `label_encoders.pkl`) were fit
independently — they are **not guaranteed to cover the same set of articles** (CF only knows
about items that appeared in at least one training transaction; CB knows about every article in
the catalog). We check this explicitly rather than assuming alignment.


In [6]:
cb_article_set = set(cb_article_ids)
cf_article_set = set(item_encoder.classes_)

only_in_cb = cb_article_set - cf_article_set
only_in_cf = cf_article_set - cb_article_set
in_both = cb_article_set & cf_article_set

print(f"Articles known to Content-Based only (no training purchases): {len(only_in_cb):,}")
print(f"Articles known to Collaborative Filtering only (shouldn't happen): {len(only_in_cf):,}")
print(f"Articles known to BOTH models: {len(in_both):,}")
print("""
Implication: for the {:,} content-only articles (e.g. brand-new or never-purchased-in-train
products), the hybrid function below will fall back to content_score alone — there is no
collaborative signal to blend in for them, by definition.
""".format(len(only_in_cb)))


Articles known to Content-Based only (no training purchases): 995
Articles known to Collaborative Filtering only (shouldn't happen): 0
Articles known to BOTH models: 104,547

Implication: for the 995 content-only articles (e.g. brand-new or never-purchased-in-train
products), the hybrid function below will fall back to content_score alone — there is no
collaborative signal to blend in for them, by definition.



---
## 2. Why Hybrid?

**Why Content-Based alone is insufficient:** it only ever compares product text/attributes — it
can't learn that customers who buy phone cases also tend to buy screen protectors (content-unrelated
items linked by real purchase behavior), and it tends to recommend near-duplicates of whatever
the user already bought (narrow, low-diversity recommendations).

**Why Collaborative Filtering alone is insufficient:** it has zero signal for brand-new users or
items with no training interactions (the cold-start problem, confirmed concretely in Section 1.3
above), and its latent factors have no human-interpretable meaning — no natural explanation for
*why* something was recommended.

**How combining them solves their weaknesses:** each model's blind spot is roughly the other
model's strength. Content-Based covers items/users collaborative filtering has never seen.
Collaborative filtering captures behavioral patterns content-based filtering structurally cannot.
Blending both means the system degrades gracefully — a user with zero purchase history still
gets content-based recommendations, and a user with rich history gets recommendations informed
by real co-purchase behavior on top of content relevance.

**Cold-start problem revisited:** for a completely new user (no purchases at all), *neither*
individual model actually works — Content-Based needs at least one purchased item to build a
profile from, and Collaborative Filtering needs training history. This is why the hybrid function
in Section 4 explicitly falls back to the **popularity baseline** for these users, rather than
trying to force a blend that has nothing to blend.

**Popularity bias:** left unchecked, both models can quietly collapse into "just recommend
what's already popular" (checked explicitly for CF in Notebook 05, Section 7). Blending with
content-based similarity — which has no inherent popularity signal at all, since it's driven
purely by textual similarity — helps counteract this, checked again for the hybrid model in
Section 7 below.

**The basic idea:**

```
Content Score
       +
Collaborative Score
       ↓
Hybrid Score
```


---
## 3. Build Hybrid Scoring

**Why normalization matters:** the content-based cosine similarity score is naturally bounded
between 0 and 1. The ALS collaborative score is a raw dot product of latent factors — it has no
fixed range, and its scale depends on the trained factors themselves. Combining them directly
would let whichever score happens to have a larger numeric range dominate the blend regardless of
`alpha` — so we min-max normalize both scores within each candidate pool before combining.

**Hybrid formula:**

```
Hybrid Score = α × Content Score + (1 - α) × Collaborative Score
```


In [7]:
import numpy as np

def get_content_score_vector(profile_vector, candidate_indices):
    """Cosine similarity between a user's content profile vector and a set of candidate items,
    computed only for the requested candidates (not the full catalog) for efficiency."""
    from sklearn.metrics.pairwise import cosine_similarity
    candidate_vectors = tfidf_matrix[candidate_indices]
    scores = cosine_similarity(profile_vector, candidate_vectors).flatten()
    return scores

def get_collaborative_score_vector(user_idx, candidate_item_indices):
    """ALS affinity score (user_factor . item_factor) for a set of candidate items —
    computed directly from the learned factors, not via the top-N recommend() call, so it
    works for ANY candidate item, not just ones ALS would have ranked highly on its own."""
    user_vector = als_model.user_factors[user_idx]
    item_vectors = als_model.item_factors[candidate_item_indices]
    scores = item_vectors @ user_vector
    return scores

def min_max_normalize(scores):
    """Scale a score array to [0, 1] within itself, so content and collaborative scores
    become directly comparable regardless of their original scale."""
    if len(scores) == 0:
        return scores
    min_val, max_val = scores.min(), scores.max()
    if max_val - min_val < 1e-9:
        return np.zeros_like(scores)  # avoid divide-by-zero when all candidates score equally
    return (scores - min_val) / (max_val - min_val)


**3.1 Experiment with alpha values**

Rather than assuming a value, we quickly evaluate a few reasonable `alpha` values on a validation
sample and pick based on measured performance (Precision@10), the same practical approach used
for ALS's hyperparameters in Notebook 05.


In [8]:
def build_user_content_profile(user_idx, train_matrix, cb_id_to_index, item_encoder):
    """Build a content 'taste profile' vector for a user by averaging the TF-IDF vectors
    of everything they purchased in training."""
    purchased_item_indices = train_matrix[user_idx].indices  # CF item_idx space
    if len(purchased_item_indices) == 0:
        return None

    purchased_article_ids = item_encoder.inverse_transform(purchased_item_indices)
    cb_indices = [cb_id_to_index[aid] for aid in purchased_article_ids if aid in cb_id_to_index]
    if not cb_indices:
        return None

    profile_vector = tfidf_matrix[cb_indices].mean(axis=0)
    return np.asarray(profile_vector)


In [11]:
def recommend_hybrid_core(user_idx, alpha, n=10, candidate_pool_size=50):
    """Core hybrid logic used both for evaluation and the final recommend_hybrid() function.
    Returns a list of (item_idx, hybrid_score, content_score, collab_score) tuples."""

    already_purchased = set(train_matrix[user_idx].indices)

    als_ids, _ = als_model.recommend(
        user_idx, train_matrix[user_idx], N=candidate_pool_size, filter_already_liked_items=True
    )

    # Content-based candidates: lets genuinely niche, non-popular items into consideration,
    # instead of only ever re-ranking a pool pre-filtered to popular items
    profile_vector = build_user_content_profile(user_idx, train_matrix, cb_id_to_index, item_encoder)
    content_candidates = []
    if profile_vector is not None:
        from sklearn.metrics.pairwise import cosine_similarity
        scores = cosine_similarity(profile_vector, tfidf_matrix).flatten()
        top_cb_indices = scores.argsort()[::-1][:candidate_pool_size]
        for cb_idx in top_cb_indices:
            aid = cb_article_ids[cb_idx]
            if aid in item_encoder.classes_:
                content_candidates.append(item_encoder.transform([aid])[0])

    # Popularity now only a small backstop (20), not half the candidate pool
    candidate_items = list(dict.fromkeys(
        list(als_ids) + content_candidates + list(popularity_ranking[:20])
    ))
    candidate_items = [i for i in candidate_items if i not in already_purchased]

    if not candidate_items:
        return []

    candidate_article_ids = item_encoder.inverse_transform(candidate_items)
    valid_pairs = [
        (item_idx, cb_id_to_index[aid])
        for item_idx, aid in zip(candidate_items, candidate_article_ids)
        if aid in cb_id_to_index
    ]
    if not valid_pairs:
        return []

    valid_item_indices, valid_cb_indices = zip(*valid_pairs)
    valid_item_indices = np.array(valid_item_indices)
    valid_cb_indices = np.array(valid_cb_indices)

    collab_scores_raw = get_collaborative_score_vector(user_idx, valid_item_indices)

    if profile_vector is not None:
        content_scores_raw = get_content_score_vector(profile_vector, valid_cb_indices)
    else:
        content_scores_raw = np.zeros(len(valid_item_indices))

    content_scores_norm = min_max_normalize(content_scores_raw)
    collab_scores_norm = min_max_normalize(collab_scores_raw)

    hybrid_scores = alpha * content_scores_norm + (1 - alpha) * collab_scores_norm

    results = list(zip(valid_item_indices, hybrid_scores, content_scores_norm, collab_scores_norm))
    results.sort(key=lambda x: x[1], reverse=True)
    return results[:n]

In [12]:
def hybrid_precision_at_10(alpha, test_df, n_sample_users=300):
    test_users = test_df["user_idx"].unique()
    sample_users = np.random.RandomState(42).choice(
        test_users, min(n_sample_users, len(test_users)), replace=False
    )

    precisions = []
    for u in sample_users:
        relevant_items = set(test_df[test_df["user_idx"] == u]["item_idx"].values)
        if not relevant_items:
            continue
        recs = recommend_hybrid_core(u, alpha=alpha, n=10)
        recommended_items = set(r[0] for r in recs)
        hits = len(recommended_items & relevant_items)
        precisions.append(hits / 10)

    return np.mean(precisions) if precisions else 0.0

test_df = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "test_interactions_encoded.parquet"))

alpha_candidates = [0.2, 0.4, 0.6, 0.8]
alpha_results = []
for a in alpha_candidates:
    p10 = hybrid_precision_at_10(a, test_df, n_sample_users=300)
    alpha_results.append({"alpha": a, "precision_at_10": p10})
    print(f"alpha={a} -> Precision@10 = {p10:.4f}")

alpha_comparison = pd.DataFrame(alpha_results)
display(alpha_comparison)

BEST_ALPHA = alpha_comparison.sort_values("precision_at_10", ascending=False).iloc[0]["alpha"]
print(f"\nSelected alpha: {BEST_ALPHA}")


alpha=0.2 -> Precision@10 = 0.0053
alpha=0.4 -> Precision@10 = 0.0050
alpha=0.6 -> Precision@10 = 0.0033
alpha=0.8 -> Precision@10 = 0.0020


,alpha,precision_at_10
0,0.2,0.005333
1,0.4,0.005000
2,0.6,0.003333
3,0.8,0.002000



Selected alpha: 0.2


---
## 4. Build Recommendation Function


In [13]:
article_lookup = processed_articles.set_index("article_id")

def recommend_hybrid(customer_id, n=10, alpha=None):
    """Full hybrid recommendation pipeline for a real customer_id.
    Handles new users (no purchase history) by falling back to popularity."""

    if alpha is None:
        alpha = BEST_ALPHA

    # New / unknown user: no row in the CF encoder at all
    if customer_id not in user_encoder.classes_:
        print(f"New user ({customer_id}) — no purchase history at all, using popularity fallback.")
        top_items = [i for i in popularity_ranking if True][:n]
        rows = []
        for item_idx in top_items:
            aid = item_encoder.inverse_transform([item_idx])[0]
            if aid not in article_lookup.index:
                continue
            meta = article_lookup.loc[aid]
            rows.append({
                "article_id": aid, "product_name": meta.get("prod_name", ""),
                "category": meta.get("product_group_name", ""),
                "content_score": None, "collaborative_score": None, "hybrid_score": None
            })
        return pd.DataFrame(rows)

    user_idx = user_encoder.transform([customer_id])[0]

    # Adaptive alpha by user activity level — see Section 5 for the reasoning
    alpha = get_adaptive_alpha(user_idx, default_alpha=alpha)

    results = recommend_hybrid_core(user_idx, alpha=alpha, n=n)

    rows = []
    for item_idx, hybrid_score, content_score, collab_score in results:
        aid = item_encoder.inverse_transform([item_idx])[0]
        if aid not in article_lookup.index:
            continue
        meta = article_lookup.loc[aid]
        rows.append({
            "article_id": aid,
            "product_name": meta.get("prod_name", ""),
            "category": meta.get("product_group_name", ""),
            "content_score": round(float(content_score), 4),
            "collaborative_score": round(float(collab_score), 4),
            "hybrid_score": round(float(hybrid_score), 4)
        })

    return pd.DataFrame(rows)


---
## 5. Handle Different User Types

Rather than always using one fixed `alpha`, we adjust it based on how much purchase history a
user has — this reflects a simple, explainable rule: **trust collaborative filtering more once
there's enough behavioral data to trust, and lean on content-based similarity when there isn't.**

- **New user (0 purchases):** neither model has anything to work with — fall back to popularity
  entirely (handled in `recommend_hybrid` above, before `alpha` even comes into play).
- **Low-activity user (few purchases, e.g. < 5):** collaborative filtering's latent factors for
  this user are based on very little data and are unreliable — weight content-based higher
  (`alpha` closer to 0.8) since a handful of purchases is still enough to build a meaningful
  content profile from.
- **Active user (enough purchases, e.g. >= 5):** collaborative filtering has real behavioral
  signal to work with — weight it higher (`alpha` closer to 0.2), since it tends to capture
  patterns content similarity alone would miss.


In [14]:
train_df = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "train_interactions_encoded.parquet"))
print("train_df loaded:", train_df.shape)

train_df loaded: (27101148, 6)


In [15]:
LOW_ACTIVITY_THRESHOLD = 5

def get_adaptive_alpha(user_idx, default_alpha):
    """Return a higher alpha (more content-based weight) for low-activity users,
    a lower alpha (more collaborative weight) for active users."""
    n_purchases = train_matrix[user_idx].nnz

    if n_purchases == 0:
        return default_alpha  # shouldn't reach here (handled earlier), but safe fallback
    elif n_purchases < LOW_ACTIVITY_THRESHOLD:
        return 0.8
    else:
        return 0.2

# Quick demonstration across the three user types
print("Adaptive alpha examples:")
for u in [train_df["user_idx"].iloc[0]]:
    n_purch = train_matrix[u].nnz
    print(f"  user_idx={u}, purchases={n_purch}, adaptive_alpha={get_adaptive_alpha(u, BEST_ALPHA)}")


Adaptive alpha examples:
  user_idx=0, purchases=19, adaptive_alpha=0.2


> **Note:** `train_df` here refers to the training interactions loaded earlier for building
> `train_matrix` — if not already in memory in this session, reload it:
> `train_df = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "train_interactions_encoded.parquet"))`


---
## 6. Evaluate the Hybrid Model

We reuse the same time-based test set and metric definitions from Notebook 05 (Precision@K,
Recall@K, NDCG@10, Hit Rate@10), now comparing **four** approaches on the identical user sample.


In [16]:
def dcg_at_k(relevance_list, k):
    relevance_list = relevance_list[:k]
    return sum((rel / np.log2(idx + 2)) for idx, rel in enumerate(relevance_list))

def ndcg_at_k(recommended_items, relevant_items, k):
    relevance = [1 if item in relevant_items else 0 for item in recommended_items[:k]]
    ideal_relevance = sorted(relevance, reverse=True)
    dcg = dcg_at_k(relevance, k)
    idcg = dcg_at_k(ideal_relevance, k)
    return dcg / idcg if idcg > 0 else 0.0

def evaluate_recommend_fn(recommend_fn, test_df, n_sample_users=1500, ks=(5, 10)):
    test_users = test_df["user_idx"].unique()
    sample_users = np.random.RandomState(42).choice(
        test_users, min(n_sample_users, len(test_users)), replace=False
    )

    metrics = {f"precision@{k}": [] for k in ks}
    metrics.update({f"recall@{k}": [] for k in ks})
    metrics["ndcg@10"] = []
    metrics["hit_rate@10"] = []

    for u in sample_users:
        relevant_items = set(test_df[test_df["user_idx"] == u]["item_idx"].values)
        if not relevant_items:
            continue

        recs_10 = recommend_fn(u, n=10)

        for k in ks:
            recs_k = recs_10[:k]
            hits = len(set(recs_k) & relevant_items)
            metrics[f"precision@{k}"].append(hits / k)
            metrics[f"recall@{k}"].append(hits / len(relevant_items))

        metrics["ndcg@10"].append(ndcg_at_k(recs_10, relevant_items, 10))
        metrics["hit_rate@10"].append(1 if len(set(recs_10) & relevant_items) > 0 else 0)

    return {metric: float(np.mean(values)) for metric, values in metrics.items() if values}


In [17]:
def popularity_recommend_fn(user_idx, n=10):
    already_purchased = set(train_matrix[user_idx].indices)
    return [i for i in popularity_ranking if i not in already_purchased][:n]

def content_recommend_fn(user_idx, n=10):
    profile_vector = build_user_content_profile(user_idx, train_matrix, cb_id_to_index, item_encoder)
    already_purchased = set(train_matrix[user_idx].indices)
    if profile_vector is None:
        return popularity_recommend_fn(user_idx, n=n)

    from sklearn.metrics.pairwise import cosine_similarity
    scores = cosine_similarity(profile_vector, tfidf_matrix).flatten()
    top_cb_indices = scores.argsort()[::-1]

    recs = []
    for cb_idx in top_cb_indices:
        aid = cb_article_ids[cb_idx]
        if aid not in item_encoder.classes_:
            continue
        item_idx = item_encoder.transform([aid])[0]
        if item_idx in already_purchased:
            continue
        recs.append(item_idx)
        if len(recs) == n:
            break
    return recs

def als_recommend_fn(user_idx, n=10):
    ids, _ = als_model.recommend(user_idx, train_matrix[user_idx], N=n, filter_already_liked_items=True)
    return list(ids)

def hybrid_recommend_fn(user_idx, n=10):
    alpha = get_adaptive_alpha(user_idx, default_alpha=BEST_ALPHA)
    results = recommend_hybrid_core(user_idx, alpha=alpha, n=n)
    return [r[0] for r in results]


In [18]:
N_EVAL_USERS = 1500

print("Evaluating all four approaches on the same user sample...")
results_popularity = evaluate_recommend_fn(popularity_recommend_fn, test_df, n_sample_users=N_EVAL_USERS)
print("Popularity done.")
results_content = evaluate_recommend_fn(content_recommend_fn, test_df, n_sample_users=N_EVAL_USERS)
print("Content-Based done.")
results_als = evaluate_recommend_fn(als_recommend_fn, test_df, n_sample_users=N_EVAL_USERS)
print("Collaborative Filtering done.")
results_hybrid = evaluate_recommend_fn(hybrid_recommend_fn, test_df, n_sample_users=N_EVAL_USERS)
print("Hybrid done.")

final_comparison = pd.DataFrame([
    {"model": "Popularity Baseline", **results_popularity},
    {"model": "Content-Based", **results_content},
    {"model": "Collaborative Filtering (ALS)", **results_als},
    {"model": "Hybrid", **results_hybrid},
])
display(final_comparison)


Evaluating all four approaches on the same user sample...
Popularity done.
Content-Based done.
Collaborative Filtering done.
Hybrid done.


,model,precision@5,precision@10,recall@5,recall@10,ndcg@10,hit_rate@10
0,Popularity Baseline,0.001600,0.001400,0.002819,0.004397,0.007759,0.013333
1,Content-Based,0.001733,0.001533,0.004833,0.007169,0.007787,0.014667
2,Collaborative Filtering (ALS),0.004800,0.003667,0.007052,0.011893,0.016670,0.032667
3,Hybrid,0.005733,0.004067,0.010586,0.014719,0.019408,0.036000


---
## 7. Analyze the Hybrid Recommendations


In [19]:
from collections import Counter

def analyze_bias(recommend_fn, test_df, n_sample_users=1000, n=10):
    test_users = test_df["user_idx"].unique()
    sample_users = np.random.RandomState(42).choice(
        test_users, min(n_sample_users, len(test_users)), replace=False
    )

    all_recs = []
    for u in sample_users:
        all_recs.extend(recommend_fn(u, n=n))

    item_counts = Counter(all_recs)
    unique_recommended = len(item_counts)
    diversity_ratio = unique_recommended / len(all_recs)

    top_100_popular = set(popularity_ranking[:100])
    popularity_bias_pct = sum(1 for i in all_recs if i in top_100_popular) / len(all_recs) * 100

    coverage_pct = unique_recommended / len(item_encoder.classes_) * 100

    return {
        "diversity_ratio": diversity_ratio,
        "popularity_bias_pct": popularity_bias_pct,
        "coverage_pct": coverage_pct
    }

bias_comparison = pd.DataFrame([
    {"model": "Popularity Baseline", **analyze_bias(popularity_recommend_fn, test_df)},
    {"model": "Content-Based", **analyze_bias(content_recommend_fn, test_df)},
    {"model": "Collaborative Filtering (ALS)", **analyze_bias(als_recommend_fn, test_df)},
    {"model": "Hybrid", **analyze_bias(hybrid_recommend_fn, test_df)},
])
display(bias_comparison)


,model,diversity_ratio,popularity_bias_pct,coverage_pct
0,Popularity Baseline,0.0015,100.00,0.014348
1,Content-Based,0.5779,1.21,5.527657
2,Collaborative Filtering (ALS),0.0888,50.84,0.849379
3,Hybrid,0.1671,41.55,1.598324


**Whether hybrid recommendations are more balanced:** compare the Hybrid row against both
individual models. A well-functioning hybrid should show **diversity and coverage between the
two individual models** (not identical to either), and **lower popularity bias than the
Popularity Baseline row** — evidence it isn't just re-deriving popularity indirectly through
either component model.


**7.1 Real examples across a few different users**

In [20]:
def show_user_example(customer_id):
    user_idx = user_encoder.transform([customer_id])[0]
    purchased_indices = train_matrix[user_idx].indices
    purchased_ids = item_encoder.inverse_transform(purchased_indices)[:5]
    purchased_names = [
        article_lookup.loc[aid, "prod_name"] for aid in purchased_ids if aid in article_lookup.index
    ]

    print(f"USER: {customer_id}")
    print(f"Previous purchases (sample): {purchased_names}")

    content_recs = content_recommend_fn(user_idx, n=5)
    content_names = [
        article_lookup.loc[item_encoder.inverse_transform([i])[0], "prod_name"]
        for i in content_recs if item_encoder.inverse_transform([i])[0] in article_lookup.index
    ]
    print(f"\nContent-Based recommendations: {content_names}")

    als_recs = als_recommend_fn(user_idx, n=5)
    als_names = [
        article_lookup.loc[item_encoder.inverse_transform([i])[0], "prod_name"]
        for i in als_recs if item_encoder.inverse_transform([i])[0] in article_lookup.index
    ]
    print(f"ALS recommendations: {als_names}")

    hybrid_df = recommend_hybrid(customer_id, n=5)
    print(f"Hybrid recommendations:")
    display(hybrid_df)
    print("\n" + "="*70 + "\n")

# Show a couple of example users with different activity levels
example_customers = user_encoder.inverse_transform(
    train_df.groupby("user_idx").size().sort_values(ascending=False).index[:2].tolist() +
    train_df.groupby("user_idx").size().sort_values().index[:1].tolist()
)
for cust in example_customers:
    show_user_example(cust)


USER: be1981ab818cf4ef6765b2ecaea7a2cbf14ccd6e8a7ee985513d9e8e53c6d91b
Previous purchases (sample): ['OP T-shirt (Idro)', 'Box 4p Tights', '3p Sneaker Socks', '3p Sneaker Socks', '4p Claw']

Content-Based recommendations: ['Nolana terry dress', 'Bansky dress', 'Marc dress', 'Nolana terry dress', 'KORA DRESS']
ALS recommendations: ['Enter treggings', 'Push Up Jegging L.W', 'Velvet scrunchie', 'Alex Jogger (J)', 'Nirvana']
Hybrid recommendations:


,article_id,product_name,category,content_score,collaborative_score,hybrid_score
0,658030001,Push Up Jegging L.W,Garment Lower body,0.5062,0.9678,0.8754
1,817353002,Nirvana,Garment Full body,0.6794,0.8988,0.8549
2,661306008,Enter treggings,Garment Lower body,0.2718,1.0000,0.8544
3,817353008,Nirvana,Garment Full body,0.6794,0.8730,0.8343
4,841260011,Jen tee,Garment Upper body,0.5565,0.8847,0.8190




USER: cd04ec2726dd58a8c753e0d6423e57716fd9ebcf2f14ed6012e7e5bea016b4d6
Previous purchases (sample): ['Anita Tank (1)', 'Jacket Slim (1)', 'Juan lace strap top', 'Tikki Pull-on TRS', 'Zelda Jumper']

Content-Based recommendations: ['Sienna dress', 'Nikita dress', 'Nolana terry dress', 'Siri dress', 'Mindy dress']
ALS recommendations: ['Bella l/s', 'H2 Lauren dress PI', 'Luna skinny RW', 'Hilton', 'Amanda Rib']
Hybrid recommendations:


,article_id,product_name,category,content_score,collaborative_score,hybrid_score
0,708428004,Bella l/s,Garment Upper body,0.6706,1.0000,0.9341
1,767799004,JUST PINK DRESS(1),Garment Full body,0.7925,0.9567,0.9239
2,819139001,H2 Lauren dress PI,Garment Full body,0.5940,0.9992,0.9182
3,708379003,Simone,Garment Upper body,0.7814,0.9281,0.8987
4,803772002,Amanda Rib,Garment Upper body,0.6210,0.9681,0.8987




USER: 0002697f519fce0a41f12929696aad2cda5b3a0524896152e62d9ac4f5abe232
Previous purchases (sample): ['Julie (1)']

Content-Based recommendations: ['Julie (1)', 'Julie (1)', 'Julie (1)', 'KENTY sweatpants SB', 'KENTY sweatpants SB']
ALS recommendations: ['Victoria RW Pull- On TRS', 'Bradley trousers', 'Long leggings update', 'Daiquiri Pull- On TRS', 'Long Leggings']
Hybrid recommendations:


,article_id,product_name,category,content_score,collaborative_score,hybrid_score
0,562498015,Julie (1),Garment Lower body,1.0000,0.7524,0.9505
1,562498011,Julie (1),Garment Lower body,0.8645,0.7830,0.8482
2,562498004,Julie (1),Garment Lower body,0.8756,0.4081,0.7821
3,803757001,Bradley trousers,Garment Lower body,0.3534,0.9389,0.4705
4,179123001,Long Leggings,Garment Lower body,0.3657,0.8869,0.4699


---
## 8. Save the Final Recommendation Engine


In [21]:
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

hybrid_config = {
    "best_alpha": float(BEST_ALPHA),
    "low_activity_threshold": LOW_ACTIVITY_THRESHOLD,
    "low_activity_alpha": 0.8,
    "active_user_alpha": 0.2,
    "candidate_pool_size": 50,
}
with open(os.path.join(MODELS_DIR, "hybrid_model_config.pkl"), "wb") as f:
    pickle.dump(hybrid_config, f)

final_comparison.to_csv(os.path.join(ARTIFACTS_DIR, "hybrid_evaluation.csv"), index=False)
bias_comparison.to_csv(os.path.join(ARTIFACTS_DIR, "hybrid_bias_analysis.csv"), index=False)
alpha_comparison.to_csv(os.path.join(ARTIFACTS_DIR, "alpha_tuning_results.csv"), index=False)

print("Saved: models/hybrid_model_config.pkl")
print("Saved: artifacts/hybrid_evaluation.csv")
print("Saved: artifacts/hybrid_bias_analysis.csv")
print("Saved: artifacts/alpha_tuning_results.csv")

print("""
What's needed to recreate hybrid recommendations later, WITHOUT retraining anything:
  - cb_artifacts/tfidf_vectorizer.pkl + tfidf_matrix.npz  (Notebook 04)
  - models/als_model.pkl + artifacts/train_matrix.npz      (Notebook 05)
  - processed_data/label_encoders.pkl, content_features.csv, processed_articles.csv (Step 3)
  - models/hybrid_model_config.pkl                          (this notebook — alpha + thresholds)
All of the above are now saved and sufficient to reload and call recommend_hybrid() fresh.
""")


Saved: models/hybrid_model_config.pkl
Saved: artifacts/hybrid_evaluation.csv
Saved: artifacts/hybrid_bias_analysis.csv
Saved: artifacts/alpha_tuning_results.csv

What's needed to recreate hybrid recommendations later, WITHOUT retraining anything:
  - cb_artifacts/tfidf_vectorizer.pkl + tfidf_matrix.npz  (Notebook 04)
  - models/als_model.pkl + artifacts/train_matrix.npz      (Notebook 05)
  - processed_data/label_encoders.pkl, content_features.csv, processed_articles.csv (Step 3)
  - models/hybrid_model_config.pkl                          (this notebook — alpha + thresholds)
All of the above are now saved and sufficient to reload and call recommend_hybrid() fresh.



---
## 9. Final Interpretation

**Which model performed best?** See the Section 6 comparison table — read the actual
Precision@10 / NDCG@10 / Hit Rate@10 values there rather than assuming; on implicit-feedback
e-commerce data, the ranking between Content-Based and ALS is not always intuitive, and the whole
point of Section 6 is to let the measured numbers answer this rather than a prior assumption.

**Did the hybrid model improve over individual models?** Compare the Hybrid row specifically
against whichever individual model scored highest. A genuine improvement here is the core
justification for the added complexity of maintaining two models instead of one — if the hybrid
doesn't clearly beat the best individual model, that's worth being honest about rather than
assuming hybrid is automatically superior.

**What happens when alpha changes?** Section 3.1's `alpha_comparison` table shows this directly —
higher alpha shifts weight toward content similarity (more thematically consistent but less
behaviorally personalized recommendations); lower alpha shifts weight toward collaborative
filtering (more behaviorally personalized but less explainable, and more exposed to cold-start
gaps for candidates content filtering could otherwise catch).

**Why does the hybrid approach work?** Because the two models fail in different situations —
content-based fails to capture behavioral co-purchase patterns, collaborative filtering fails on
cold-start users/items. Blending means the system's weak points are covered by the other model's
strengths, rather than the system inheriting one model's single point of failure.

**Remaining limitations:**
- Still doesn't solve *pure* cold-start (zero-purchase-history) users beyond a popularity fallback
- `alpha` and the activity thresholds were tuned on a specific evaluation sample — real production
  deployment would want ongoing monitoring and re-tuning as behavior shifts over time
- No use of customer demographic features (`customer_features.csv` from Step 3) in the current
  blend — a natural next enhancement, not implemented here
- Evaluation was done on a sample of users for computational feasibility, not the full test set

---

**Next Notebook: Final Evaluation & Recommendation System Analysis.**
No FastAPI, Streamlit, or deployment work is built in this notebook — that comes later.
